# Practice 3: Router with tests (expense approval)

**What this notebook does:** builds the expense-approval graph, then shows how to write `pytest` tests for it.

The route is: amount **under 50** is auto-approved, **50 to 499** goes to a manager, **500 and above** goes to a director. The manager and director steps pause the graph and wait for a yes or no.

**Why the code is written to files first.** `pytest` can only import code from `.py` files, not from notebook cells. So the two code cells marked `%%writefile` create `expense_graph.py` (the logic) and `test_expense_graph.py` (the tests) next to this notebook. Everything else in the notebook imports from those files. This is the normal pattern: logic in a module, a notebook or script to play with it, tests beside it.

**Three fixes compared with the first version**
1. The router is now a *pure function*. It only looks at the state and returns a name. It no longer prints or edits the state, because edits made inside a router are silently thrown away.
2. Every node has an explicit edge to `END`, and the conditional edge has an explicit map of name to node.
3. Every run uses a fresh `thread_id`. Reusing one makes the new run continue the old state.

## Step 1. The module (logic only)

In [1]:
%%writefile expense_graph.py
from typing import TypedDict, Literal, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt

AUTO_LIMIT = 50        # below this: approved automatically
MANAGER_LIMIT = 500    # below this: a manager decides; at or above: a director


class State(TypedDict):
    expense: int
    comment: Annotated[list[str], operator.add]


# ---- the router: a PURE function. No printing, no changing state. ----
def route_by_amount(state: State) -> Literal["auto_approve", "manager", "director"]:
    amount = state["expense"]
    if amount < 0:
        raise ValueError(f"expense cannot be negative: {amount}")
    if amount < AUTO_LIMIT:
        return "auto_approve"
    if amount < MANAGER_LIMIT:
        return "manager"
    return "director"


# ---- nodes: take the state, return only the changes ----
def start_check(state: State) -> dict:
    return {"comment": [f"Checking expense of {state['expense']}"]}


def auto_approve(state: State) -> dict:
    return {"comment": ["Expense auto approved"]}


def manager(state: State) -> dict:
    answer = interrupt({"question": f"Manager: approve {state['expense']}? yes or no"})
    verdict = "approved" if answer == "yes" else "not approved"
    return {"comment": [f"Expense {verdict} by manager"]}


def director(state: State) -> dict:
    answer = interrupt({"question": f"Director: approve {state['expense']}? yes or no"})
    verdict = "approved" if answer == "yes" else "not approved"
    return {"comment": [f"Expense {verdict} by director"]}


def build_graph(checkpointer=None):
    builder = StateGraph(State)
    builder.add_node("start_check", start_check)
    builder.add_node("auto_approve", auto_approve)
    builder.add_node("manager", manager)
    builder.add_node("director", director)

    builder.add_edge(START, "start_check")
    builder.add_conditional_edges(
        "start_check",
        route_by_amount,
        {"auto_approve": "auto_approve", "manager": "manager", "director": "director"},
    )
    builder.add_edge("auto_approve", END)
    builder.add_edge("manager", END)
    builder.add_edge("director", END)
    return builder.compile(checkpointer=checkpointer)

Writing expense_graph.py


## Step 2. Try the router by hand

Because the router is pure, you can call it like any normal function: no graph, no pause, no memory. This is exactly what makes it easy to test.

In [2]:
from expense_graph import route_by_amount, build_graph, State

for amount in [0, 49, 50, 499, 500, 10_000]:
    print(f"{amount:>6} -> {route_by_amount({'expense': amount, 'comment': []})}")

     0 -> auto_approve
    49 -> auto_approve
    50 -> manager
   499 -> manager
   500 -> director
 10000 -> director


## Step 3. Look at the graph
The dashed lines are the conditional choices made by the router. (`draw_mermaid_png()` needs a network call, so this prints the text version, which you can paste into any Mermaid viewer.)

In [3]:
print(build_graph().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	start_check(start_check)
	auto_approve(auto_approve)
	manager(manager)
	director(director)
	__end__([<p>__end__</p>]):::last
	__start__ --> start_check;
	start_check -.-> auto_approve;
	start_check -.-> director;
	start_check -.-> manager;
	auto_approve --> __end__;
	director --> __end__;
	manager --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 4. Run it, with a fresh thread each time
A *thread id* is the name of one saved conversation of the graph. Same id means the same state carries on. Use `uuid` so each run is its own thread.

In [4]:
import uuid
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

graph = build_graph(InMemorySaver())

def new_config():
    return {"configurable": {"thread_id": str(uuid.uuid4())}}

# small amount: runs straight through
print(graph.invoke({"expense": 49, "comment": []}, new_config()))

{'expense': 49, 'comment': ['Checking expense of 49', 'Expense auto approved']}


In [5]:
# medium amount: pauses at the manager, then we resume with "yes"
cfg = new_config()
paused = graph.invoke({"expense": 50, "comment": []}, cfg)
print("PAUSED :", paused["__interrupt__"][0].value)
print("RESUMED:", graph.invoke(Command(resume="yes"), cfg))

PAUSED : {'question': 'Manager: approve 50? yes or no'}
RESUMED: {'expense': 50, 'comment': ['Checking expense of 50', 'Expense approved by manager']}


## Step 5. The surprise from the first version: reusing a thread

The first notebook reused the thread id `t5` for a second run. The second run then started with the first run's comments already in the state, because the same thread keeps its saved state and the `comment` reducer adds to it. Here it is on purpose:

In [6]:
cfg = new_config()
graph.invoke({"expense": 49, "comment": []}, cfg)
second = graph.invoke({"expense": 49, "comment": []}, cfg)
print(len(second["comment"]), "comments, not 2:")
for line in second["comment"]:
    print("  ", line)

4 comments, not 2:
   Checking expense of 49
   Expense auto approved
   Checking expense of 49
   Expense auto approved


## Step 6. Writing the tests

Four pytest ideas, all used in the file below:

- **A test is a function whose name starts with `test_` and that uses `assert`.** If every assert is true, the test passes.
- **`@pytest.mark.parametrize`** runs one test body for many inputs. Here it checks the router at the edges: 49 vs 50 and 499 vs 500. Bugs like `<` versus `<=` live exactly at those edges.
- **`pytest.raises`** checks that the right error is raised. A test for bad input should expect the error, not avoid it.
- **Fixtures (`@pytest.fixture`)** build the things a test needs. Any test that lists `graph` or `config` as an argument gets a fresh one, so tests cannot affect each other. The fresh `thread_id` per test fixes the reuse problem from Step 5.

The tests are in three groups: the pure router (no graph), the whole graph through its three paths (including pause and resume), and threads and memory.

In [7]:
%%writefile test_expense_graph.py
import copy
import uuid

import pytest
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from expense_graph import build_graph, route_by_amount


# ---------------------------------------------------------------- fixtures
@pytest.fixture
def graph():
    """A fresh graph with its own memory for every test."""
    return build_graph(InMemorySaver())


@pytest.fixture
def config():
    """A brand-new thread id for every test, so tests cannot affect each other."""
    return {"configurable": {"thread_id": str(uuid.uuid4())}}


def run(graph, config, amount):
    return graph.invoke({"expense": amount, "comment": []}, config)


def visited(graph, config, amount):
    """Names of the nodes that ran, in order."""
    names = []
    for chunk in graph.stream({"expense": amount, "comment": []}, config, stream_mode="updates"):
        names.extend(k for k in chunk if k != "__interrupt__")
    return names


# ------------------------------------------------- 1. the router (no graph)
@pytest.mark.parametrize(
    "amount, expected",
    [
        (0, "auto_approve"),
        (49, "auto_approve"),    # last value below the first limit
        (50, "manager"),         # first value at the first limit
        (499, "manager"),        # last value below the second limit
        (500, "director"),       # first value at the second limit
        (10_000, "director"),
    ],
)
def test_router_picks_the_right_branch(amount, expected):
    assert route_by_amount({"expense": amount, "comment": []}) == expected


def test_router_rejects_negative_amounts():
    with pytest.raises(ValueError, match="negative"):
        route_by_amount({"expense": -1, "comment": []})


def test_router_does_not_change_the_state():
    state = {"expense": 120, "comment": ["earlier"]}
    before = copy.deepcopy(state)
    route_by_amount(state)
    assert state == before


# ------------------------------------------ 2. the whole graph, three paths
def test_small_expense_is_auto_approved_without_pausing(graph, config):
    result = run(graph, config, 49)
    assert "__interrupt__" not in result
    assert result["comment"] == ["Checking expense of 49", "Expense auto approved"]


def test_manager_path_pauses_then_approves(graph, config):
    paused = run(graph, config, 50)
    assert "Manager" in paused["__interrupt__"][0].value["question"]

    final = graph.invoke(Command(resume="yes"), config)
    assert final["comment"][-1] == "Expense approved by manager"


def test_manager_path_can_reject(graph, config):
    run(graph, config, 200)
    final = graph.invoke(Command(resume="no"), config)
    assert final["comment"][-1] == "Expense not approved by manager"


def test_large_expense_goes_to_the_director(graph, config):
    paused = run(graph, config, 500)
    assert "Director" in paused["__interrupt__"][0].value["question"]
    final = graph.invoke(Command(resume="yes"), config)
    assert final["comment"][-1] == "Expense approved by director"


def test_path_taken_is_exactly_what_we_expect(graph, config):
    assert visited(graph, config, 49) == ["start_check", "auto_approve"]


# ------------------------------------------------- 3. threads and memory
def test_different_threads_do_not_share_comments(graph):
    a = {"configurable": {"thread_id": "A"}}
    b = {"configurable": {"thread_id": "B"}}
    run(graph, a, 49)
    result_b = run(graph, b, 49)
    assert result_b["comment"] == ["Checking expense of 49", "Expense auto approved"]


def test_reusing_a_thread_keeps_the_old_comments(graph, config):
    """Documents a behaviour that surprised us: the same thread_id continues the old state."""
    run(graph, config, 49)
    second = run(graph, config, 49)
    assert len(second["comment"]) == 4

Writing test_expense_graph.py


## Step 7. Run the tests
`-v` lists each test by name. Run this after every change to the module.

In [8]:
!python -m pytest -v --no-header -p no:cacheprovider

============================= test session starts ==============================
collecting ... 


collected 15 items                                                             

test_expense_graph.py::test_router_picks_the_right_branch[0-auto_approve] PASSED [  6%]
test_expense_graph.py::test_router_picks_the_right_branch[49-auto_approve] PASSED [ 13%]
test_expense_graph.py::test_router_picks_the_right_branch[50-manager] PASSED [ 20%]
test_expense_graph.py::test_router_picks_the_right_branch[499-manager] PASSED [ 26%]
test_expense_graph.py::test_router_picks_the_right_branch[500-director] PASSED [ 33%]
test_expense_graph.py::test_router_picks_the_right_branch[10000-director] PASSED [ 40%]
test_expense_graph.py::test_router_rejects_negative_amounts PASSED       [ 46%]
test_expense_graph.py::test_router_does_not_change_the_state PASSED      [ 53%]
test_expense_graph.py::test_small_expense_is_auto_approved_without_pausing 

PASSED [ 60%]
test_expense_graph.py::test_manager_path_pauses_then_approves PASSED     [ 66%]
test_expense_graph.py::test_manager_path_can_reject PASSED               [ 73%]
test_expense_graph.py::test_large_expense_goes_to_the_director PASSED    [ 80%]
test_expense_graph.py::test_path_taken_is_exactly_what_we_expect PASSED  [ 86%]
test_expense_graph.py::test_different_threads_do_not_share_comments PASSED [ 93%]
test_expense_graph.py::test_reusing_a_thread_keeps_the_old_comments 

PASSED [100%]

============================== 15 passed in 0.29s ==============================


## Step 8. Check that the tests can actually fail

A test you have never seen fail may not be testing anything. Here is a router written the way the first version was (it changes the state), and one with a `<=` slip. The same checks the tests make are applied to them.

In [9]:
import copy

def bad_router_mutates(state):
    state["comment"].append("routing...")        # editing the state inside a router
    return route_by_amount(state)

def bad_router_off_by_one(state):
    return "auto_approve" if state["expense"] <= 50 else route_by_amount(state)

# check 1: "router does not change the state"
s = {"expense": 120, "comment": []}
before = copy.deepcopy(s)
bad_router_mutates(s)
print("mutating router leaves state unchanged?", s == before)

# check 2: "50 goes to the manager"
print("off-by-one router sends 50 to:", bad_router_off_by_one({"expense": 50, "comment": []}),
      "(expected manager)")

mutating router leaves state unchanged? False
off-by-one router sends 50 to: auto_approve (expected manager)


## Your turn

1. Restart the kernel and run all cells. Everything above should pass.
2. Add a fourth route: expenses of 5000 and above go to a `cfo` node. Write the failing router test **first**, then change the code until it passes.
3. Add a test that rejecting at the manager does not mention the director anywhere in `comment`.
4. In `stumbles.txt`, note anything you hesitated on: "what does a fixture do?", "why must a router be pure?", "what does the same thread_id do?".